In [1]:
import pydough

#%load_ext pydough.jupyter_extensions
%reload_ext pydough.jupyter_extensions

#Necessary for comparison
import pandas as pd
from pandas.testing import assert_frame_equal, assert_series_equal
import re
import eval
import datetime

import collections
import numpy as np
import sqlite3 as sql
import os

In [15]:
#YOUR .SQL FILE TO CREATE THE DATABASE, COPY IT TO THIS FOLDER.
SQL_path = 'databases/Defog/init_defog.sql'

#METADATA FOR THE GRAPH .JSON
metadata_path = "metadata/Defog/Ewallet_graph.json"

#GRAPH NAME
graph_name = "Ewallet"

#DESIRED DATABASE NAME
DB_name = "notebookTest.db"



with open(SQL_path, 'r') as sql_file:
    sql_script = sql_file.read()

os.remove(DB_name)
connection = sql.connect(DB_name)
cursor = connection.cursor()
cursor.executescript(sql_script)

pydough.active_session.load_metadata_graph(metadata_path, graph_name)
pydough.active_session.connect_database("sqlite", database=DB_name)

DatabaseContext(connection=<pydough.database_connectors.database_connector.DatabaseConnection object at 0x750cf3180680>, dialect=<DatabaseDialect.SQLITE: 'sqlite'>)

In [1]:
tested_file, tested_df = eval.compare_output("evalNotebookFiles/", "evalNotebookFiles/noMatch.csv", ".", ".")

AttributeError: 'builtin_function_or_method' object has no attribute 'compare_output'

In [22]:
query = '''
WITH user_session_duration AS (SELECT u.uid, s.session_start_ts, s.session_end_ts FROM users AS u JOIN user_sessions AS s ON u.uid = s.user_id WHERE s.session_start_ts >= '2023-06-01' AND s.session_end_ts < '2023-06-08') SELECT uid, SUM(strftime('%s', session_end_ts) - strftime('%s', session_start_ts)) AS total_duration FROM user_session_duration GROUP BY uid ORDER BY total_duration DESC;
'''

sql_output = pd.read_sql_query(query, connection)
sql_output

,uid,total_duration
0,1,5018
1,6,4205
2,8,3640
3,2,3224
4,4,2736
5,5,2098
6,10,1843
7,9,1797
8,7,1518
9,3,1370


In [21]:
%%pydough

user_session_durations = Users.CALCULATE(
    user_id=uid,
    total_duration=ROUND(SUM(
        sessions.WHERE(
            (session_start_ts >= '2023-06-01') & (session_start_ts < '2023-06-08')
        ).CALCULATE(
            duration_in_seconds=DATEDIFF("seconds", session_start_ts, session_end_ts)
        ).duration_in_seconds
    ), 0)
).WHERE(
    total_duration > 0
).ORDER_BY(
    total_duration.DESC()
)

result = pydough.to_df(user_session_durations)
print(result)

DATEDIFF unsupported for 'DAYS'.


    user_id  total_duration
0         6          4205.0
1         2          3034.0
2         4          2736.0
3         8          2735.0
4         1          2715.0
5         1          2113.0
6         5          2098.0
7        10          1843.0
8         9          1797.0
9         7          1518.0
10        8           905.0
11        3           748.0
12        1           648.0
13        3           622.0
14        1           190.0
15        2           190.0


In [ ]:
%%pydough


selected_sessions = sessions.WHERE(
        (session_start >= "2023-06-01") & (session_end < "2023-06-08")
    ).CALCULATE(duration=DATEDIFF("seconds", session_start_ts, session_end_ts))
result = (
    Users.WHERE(HAS(selected_sessions))
    .CALCULATE(uid=uid, total_duration=SUM(selected_sessions.duration))
    .ORDER_BY(total_duration.DESC())
)
result = pydough.to_df(result)
print(result)

PyDoughQDAGException: Unrecognized term of simple table collection 'UserSessions' in graph 'Ewallet': 'session_start'